# Blooming Peccioli — Build Scenarios
Legge gli shapefile di fermate e grafo stradale e produce `weekday.json`, `weekend.json`, `eventday.json`
da copiare in `data/scenarios/` nel repo GitHub.

In [1]:
# ── DIPENDENZE ──────────────────────────────────────────────────────────────
# pip install geopandas networkx shapely numpy pyproj

import geopandas as gpd
import networkx as nx
import numpy as np
import json, math, random
from shapely.geometry import Point
from pyproj import Transformer

In [11]:
# ── PARAMETRI — modifica questi percorsi ────────────────────────────────────

STOPS_SHP   = "C:/Users/m.kroetz/OneDrive - Systematica/Valdera/06_WORK/GIS/fermate bacini/fermate_bacini_snapped.shp"      # shapefile fermate (CRS 4326)
NETWORK_SHP = "C:/Users/m.kroetz/OneDrive - Systematica/Valdera/06_WORK/Database/GIS/osm_network/edges_travel_time.shp"    # shapefile grafo strade (archi, CRS 4326)

OUT_DIR = r"C:\Users\m.kroetz\OneDrive - Systematica\Valdera\repo\blooming_peccioli\data\scenarios"      # dove scrivere i JSON

SPEED_MS = 25 * 1000 / 3600          # 25 km/h in m/s
RANDOM_SEED = 42                     # ripetibilità delle attivazioni on-demand

# Orari in secondi da mezzanotte
WEEKDAY_PEAK_AM   = 8  * 3600   # 08:00
WEEKDAY_PEAK_PM   = 18 * 3600   # 18:00
WEEKEND_PEAK_AM   = 10 * 3600   # 10:00
WEEKEND_PEAK_PM   = 20 * 3600   # 20:00
EVENT_TRIP1       = 12 * 3600   # 12:00 peccioli_1 → belvedere_5
EVENT_TRIP2       = 17 * 3600   # 17:00 belvedere_5 → peccioli_1

# Colori veicoli
COLORS = {
    "scheduled":  "#ED4347",
    "peccioli":   "#FFC000",
    "forcoli":    "#41CE69",
    "fabbrica":   "#BC7CFF",
    "belvedere":  "#0875F9",
    "rosa":       "#FD6BC9",
    "ghizzano":   "#FF9900",
}

In [12]:
# ── CARICA SHAPEFILE ────────────────────────────────────────────────────────

stops_gdf   = gpd.read_file(STOPS_SHP).to_crs(epsg=4326)
network_gdf = gpd.read_file(NETWORK_SHP).to_crs(epsg=4326)

print(f"Fermate: {len(stops_gdf)} | Archi rete: {len(network_gdf)}")
print("Colonne fermate:", list(stops_gdf.columns))
print("Colonne rete:",    list(network_gdf.columns))
stops_gdf.head()

Fermate: 34 | Archi rete: 104797
Colonne fermate: ['id', 'petalo', 'numero', 'pop', 'network_no', 'snap_dista', 'po_andata', 'po_ritorno', 'geometry']
Colonne rete: ['u', 'v', 'key', 'osmid', 'highway', 'lanes', 'maxspeed', 'name', 'oneway', 'ref', 'reversed', 'length', 'tunnel', 'bridge', 'speed_kph', 'travel_tim', 'travel_t_1', 'junction', 'access', 'service', 'width', 'in', 'geometry']


,id,petalo,numero,pop,network_no,snap_dista,po_andata,po_ritorno,geometry
0,rosa_5,rosa,5,97,1673959043,127.414080,4.0,5.0,POINT (10.68837 43.52518)
1,rosa_4,rosa,4,275,500892271,11.048256,5.0,4.0,POINT (10.6841 43.52668)
2,rosa_3,rosa,3,248,1673958956,0.000585,6.0,3.0,POINT (10.67855 43.52478)
3,rosa_2,rosa,2,293,500892217,38.081280,7.0,2.0,POINT (10.67432 43.5262)
4,rosa_1,rosa,1,6,1673959263,11.390699,8.0,1.0,POINT (10.66947 43.52922)


In [13]:
# ── COSTRUISCI GRAFO NETWORKX ────────────────────────────────────────────────
# Archi in metri (lunghezza calcolata dalla geometria in proiezione metrica)

# Convertiamo in UTM 32N per avere distanze in metri
net_utm = network_gdf.to_crs(epsg=32632)

G = nx.DiGraph()

for _, row in net_utm.iterrows():
    u = row["u"]
    v = row["v"]
    length = row.geometry.length  # metri
    G.add_edge(u, v, length=length)
    if not row.get("oneway", False):  # se non è senso unico aggiungiamo il contrario
        G.add_edge(v, u, length=length)

print(f"Grafo: {G.number_of_nodes()} nodi, {G.number_of_edges()} archi")

Grafo: 45754 nodi, 103171 archi


In [14]:
# ── AGGIUNGI COORDINATE AI NODI ──────────────────────────────────────────────
# Costruiamo un dict node_id → (lat, lon) dai nodi toccati dagli archi.
# Usiamo le coordinate dei punti iniziali/finali della geometria degli archi.

node_coords_utm = {}  # node_id → (easting, northing) in UTM 32N

for _, row in net_utm.iterrows():
    coords = list(row.geometry.coords)
    node_coords_utm[row["u"]] = coords[0]   # primo punto dell'arco
    node_coords_utm[row["v"]] = coords[-1]  # ultimo punto dell'arco

# Convertiamo in lat/lon per i waypoint del frontend
transformer = Transformer.from_crs("epsg:32632", "epsg:4326", always_xy=True)

node_latlon = {}  # node_id → (lat, lon)
for nid, (x, y) in node_coords_utm.items():
    lon, lat = transformer.transform(x, y)
    node_latlon[nid] = (lat, lon)

print(f"Coordinate nodi: {len(node_latlon)}")

Coordinate nodi: 45754


In [15]:
# ── FUNZIONI HELPER ──────────────────────────────────────────────────────────

def find_nearest_node(lat, lon, node_latlon_dict):
    """Trova il nodo del grafo più vicino a una coordinata."""
    best_node = None
    best_dist = float("inf")
    for nid, (nlat, nlon) in node_latlon_dict.items():
        d = math.hypot(lat - nlat, lon - nlon)
        if d < best_dist:
            best_dist = d
            best_node = nid
    return best_node


def route_time_and_waypoints(G, node_latlon, start_node, end_node, speed_ms):
    """
    Calcola shortest path sul grafo tra due nodi.
    Restituisce (travel_time_seconds, [(lat, lon), ...]) lungo il percorso.
    """
    try:
        path = nx.shortest_path(G, start_node, end_node, weight="length")
    except nx.NetworkXNoPath:
        print(f"  ⚠️  Nessun percorso tra {start_node} e {end_node}, uso linea retta")
        # fallback: linea retta
        c0 = node_latlon[start_node]
        c1 = node_latlon[end_node]
        dist = math.hypot(
            (c0[0]-c1[0]) * 111320,
            (c0[1]-c1[1]) * 111320 * math.cos(math.radians(c0[0]))
        )
        return dist / speed_ms, [c0, c1]

    coords = [node_latlon[n] for n in path]
    total_length = sum(
        G[path[i]][path[i+1]]["length"] for i in range(len(path)-1)
    )
    travel_time = total_length / speed_ms
    return travel_time, coords


def build_timed_waypoints(G, node_latlon, stop_list_latlon, t_start, speed_ms):
    """
    Dato un elenco ordinato di fermate (lat, lon) e un orario di partenza,
    restituisce una lista di waypoint [{t, lat, lng}, ...].
    Include i nodi intermedi del percorso per animazione fluida.
    """
    waypoints = []
    t = t_start
    prev_node = find_nearest_node(*stop_list_latlon[0], node_latlon)
    waypoints.append({"t": int(t), "lat": stop_list_latlon[0][0], "lng": stop_list_latlon[0][1]})

    for stop_lat, stop_lon in stop_list_latlon[1:]:
        curr_node = find_nearest_node(stop_lat, stop_lon, node_latlon)
        travel_time, path_coords = route_time_and_waypoints(G, node_latlon, prev_node, curr_node, speed_ms)
        # Aggiungi punti intermedi per rendering fluido (ogni ~30 sec di viaggio)
        n_intermediate = max(2, int(travel_time / 30))
        for i in range(1, len(path_coords)):
            # interpolazione lineare sull'arco
            frac = i / (len(path_coords) - 1) if len(path_coords) > 1 else 1
            t_wp = t + travel_time * frac
            waypoints.append({"t": int(t_wp), "lat": path_coords[i][0], "lng": path_coords[i][1]})
        t += travel_time
        prev_node = curr_node

    return waypoints, t  # t finale = orario di arrivo all'ultima fermata

In [16]:
# ── ORGANIZZA FERMATE PER BACINO ─────────────────────────────────────────────

# Dal campo 'id' tipo 'rosa_1', 'belvedere_5', 'peccioli_1' ecc.
# Assumiamo colonna 'id' e 'po_andata' nel shapefile.

def bacino_name(stop_id):
    return "_".join(stop_id.split("_")[:-1])  # 'rosa_1' → 'rosa'

def stop_number(stop_id):
    return int(stop_id.split("_")[-1])  # 'rosa_1' → 1

stops_gdf["bacino"]  = stops_gdf["id"].apply(bacino_name)
stops_gdf["stop_num"] = stops_gdf["id"].apply(stop_number)
stops_gdf["lat"] = stops_gdf.geometry.y
stops_gdf["lon"] = stops_gdf.geometry.x

# Dizionario bacino → fermate ordinate per po_andata
bacini = {}
for bacino, group in stops_gdf.groupby("bacino"):
    sorted_stops = group.sort_values("po_andata").reset_index(drop=True)
    bacini[bacino] = list(zip(sorted_stops["lat"], sorted_stops["lon"], sorted_stops["id"]))
    
# pila_1 appartiene logicamente al bacino forcoli
if "pila" in bacini:
    forcoli_stops = bacini.get("forcoli", []) + bacini["pila"]
    forcoli_stops = sorted(
        forcoli_stops,
        key=lambda s: stops_gdf[stops_gdf["id"] == s[2]]["po_andata"].values[0]
    )
    bacini["forcoli"] = forcoli_stops
    del bacini["pila"]    

print("Bacini trovati:", list(bacini.keys()))

# Fermata hub principale
hub = stops_gdf[stops_gdf["id"] == "peccioli_1"].iloc[0]
HUB = (hub["lat"], hub["lon"])
print(f"Hub peccioli_1: {HUB}")

# Fermata belvedere_5 per giornata evento
bel5 = stops_gdf[stops_gdf["id"] == "belvedere_4"].iloc[0]
BEL5 = (bel5["lat"], bel5["lon"])
print(f"Belvedere_5: {BEL5}")

Bacini trovati: ['belvedere', 'fabbrica', 'forcoli', 'ghizzano', 'peccioli', 'pontedera', 'rosa']
Hub peccioli_1: (np.float64(43.5466074), np.float64(10.714509))
Belvedere_5: (np.float64(43.56521789999999), np.float64(10.8078432))


In [17]:
# ── FERMATE SCHEDULED SERVICE ────────────────────────────────────────────────
# Ordine andata: peccioli_1 → pila_1 → forcoli_1 → pontedera_2 → pontedera_1
# Ordine ritorno: inverso

SCHEDULED_ORDER_IDS = ["peccioli_1", "peccioli_8", "pila_1", "forcoli_1", "pontedera_2", "pontedera_1"]

scheduled_coords = []
for sid in SCHEDULED_ORDER_IDS:
    row = stops_gdf[stops_gdf["id"] == sid].iloc[0]
    scheduled_coords.append((row["lat"], row["lon"]))

print("Fermate scheduled:", SCHEDULED_ORDER_IDS)

Fermate scheduled: ['peccioli_1', 'peccioli_8', 'pila_1', 'forcoli_1', 'pontedera_2', 'pontedera_1']


In [18]:
# ── BACINI ON-DEMAND (esclusi pontedera, pila, scheduled-only) ───────────────

ON_DEMAND_BACINI = ["peccioli", "forcoli", "fabbrica", "belvedere", "rosa", "ghizzano"]

# Il punto 'più lontano' = fermata con po_andata massimo nel bacino
def get_furthest_stop(bacino_name):
    stops = bacini[bacino_name]
    return stops[-1][:2]  # (lat, lon) dell'ultima in po_andata order

for b in ON_DEMAND_BACINI:
    if b in bacini:
        print(f"{b}: {len(bacini[b])} fermate, più lontana → {bacini[b][-1][2]}")
    else:
        print(f"⚠️  bacino '{b}' NON trovato nello shapefile — controlla il campo 'id'")

peccioli: 8 fermate, più lontana → peccioli_8
forcoli: 2 fermate, più lontana → forcoli_1
fabbrica: 5 fermate, più lontana → fabbrica_5
belvedere: 4 fermate, più lontana → belvedere_4
rosa: 7 fermate, più lontana → rosa_1
ghizzano: 6 fermate, più lontana → ghizzano_6


In [19]:
# ── FUNZIONE: costruisce la timeline di un veicolo scheduled ─────────────────
# Due veicoli: A parte da peccioli_1, B parte da pontedera_1, in direzioni opposte.
# Entrambi fanno andata+ritorno nella stessa corsa peak.

def build_scheduled_vehicle(vehicle_id, label, depart_time, stops_latlon, reverse=False):
    """
    reverse=False → andata (peccioli→pontedera)
    reverse=True  → ritorno (pontedera→peccioli)
    """
    route = list(reversed(stops_latlon)) if reverse else stops_latlon
    waypoints, t_end = build_timed_waypoints(G, node_latlon, route, depart_time, SPEED_MS)
    return {
        "id": vehicle_id,
        "label": label,
        "color": COLORS["scheduled"],
        "waypoints": waypoints
    }, t_end

print("Funzione build_scheduled_vehicle pronta")

Funzione build_scheduled_vehicle pronta


In [20]:
# ── FUNZIONE: costruisce la timeline di un veicolo on-demand ─────────────────
#
# Struttura giornata (weekday esempio):
#   1. Mattina: hub → furthest (arriva in tempo per peak_am)
#   2. On-demand intermedio: giro random tra fermate del bacino
#   3. Sera: rientro all'hub (quando scheduled arriva) → furthest

def build_ondemand_vehicle(bacino, peak_am, peak_pm, seed_offset=0):
    rng = random.Random(RANDOM_SEED + seed_offset)
    stops = bacini[bacino]  # [(lat, lon, id), ...] ordinati per po_andata
    furthest = stops[-1][:2]
    all_latlon = [s[:2] for s in stops]

    waypoints = []

    # ── 1. MATTINA: hub → furthest, arrivo a peak_am ──────────────────────
    node_hub = find_nearest_node(*HUB, node_latlon)
    node_far = find_nearest_node(*furthest, node_latlon)
    travel_to_far, _ = route_time_and_waypoints(G, node_latlon, node_hub, node_far, SPEED_MS)
    depart_morning = peak_am - travel_to_far

    wps_morning, t_at_far = build_timed_waypoints(
        G, node_latlon, [HUB, furthest], depart_morning, SPEED_MS
    )
    waypoints.extend(wps_morning)
    t_current = t_at_far

    # ── 2. ON-DEMAND INTERMEDIO: giri random tra fermate del bacino ────────
    # Generiamo 3-6 mini-corse da t_at_far fino a ~peak_pm - 90min
    t_limit_pm = peak_pm - 5400  # finisce 90 min prima del rientro
    max_stops_per_run = max(2, len(all_latlon) // 2)

    while t_current < t_limit_pm:
        # attesa breve prima di ogni corsa (10-20 min)
        wait = rng.randint(600, 1200)
        t_current += wait
        if t_current >= t_limit_pm:
            break

        # sceglie 2-max_stops fermate random, ordinate per po_andata
        n_sel = rng.randint(2, max_stops_per_run)
        selected = sorted(rng.sample(all_latlon, min(n_sel, len(all_latlon))),
                          key=lambda s: all_latlon.index(s))

        # parte dalla posizione attuale (ultima fermata raggiunta)
        last_pos = waypoints[-1]
        route = [(last_pos["lat"], last_pos["lng"])] + selected

        wps_run, t_end_run = build_timed_waypoints(
            G, node_latlon, route, t_current, SPEED_MS
        )
        if t_end_run > t_limit_pm:
            break
        waypoints.extend(wps_run[1:])  # evita duplicato del punto di partenza
        t_current = t_end_run

    # ── 3. SERA: rientro hub → furthest ───────────────────────────────────
    # Il veicolo arriva all'hub quando lo scheduled arriva da pontedera.
    # Calcoliamo quando scheduled_B (che parte da pontedera_1 a peak_pm) arriva a peccioli_1.
    node_pont = find_nearest_node(*scheduled_coords[-1], node_latlon)
    node_hub_g = find_nearest_node(*HUB, node_latlon)
    travel_sched_return, _ = route_time_and_waypoints(G, node_latlon, node_pont, node_hub_g, SPEED_MS)
    t_hub_arrival = peak_pm + travel_sched_return  # quando lo scheduled arriva all'hub

    # Il bacino deve essere all'hub in quel momento: calcoliamo partenza dal punto attuale
    last_pos = waypoints[-1]
    node_last = find_nearest_node(last_pos["lat"], last_pos["lng"], node_latlon)
    travel_to_hub, _ = route_time_and_waypoints(G, node_latlon, node_last, node_hub_g, SPEED_MS)
    t_depart_to_hub = t_hub_arrival - travel_to_hub

    if t_depart_to_hub > t_current:  # se c'è tempo libero, il veicolo aspetta
        waypoints.append({"t": int(t_depart_to_hub), "lat": last_pos["lat"], "lng": last_pos["lng"]})

    wps_to_hub, _ = build_timed_waypoints(
        G, node_latlon, [(last_pos["lat"], last_pos["lng"]), HUB], t_depart_to_hub, SPEED_MS
    )
    waypoints.extend(wps_to_hub[1:])

    # Hub → furthest per raccogliere passeggeri dalla sera
    wps_evening, _ = build_timed_waypoints(
        G, node_latlon, [HUB, furthest], t_hub_arrival, SPEED_MS
    )
    waypoints.extend(wps_evening[1:])

    return {
        "id": bacino,
        "label": bacino.capitalize(),
        "color": COLORS[bacino],
        "waypoints": waypoints
    }

print("Funzione build_ondemand_vehicle pronta")

Funzione build_ondemand_vehicle pronta


In [21]:
# ── SCENARIO WEEKDAY ─────────────────────────────────────────────────────────

peak_am = WEEKDAY_PEAK_AM
peak_pm = WEEKDAY_PEAK_PM

vehicles_weekday = []

# Scheduled A: peccioli_1 → pontedera_1 alle 8:00
veh_sched_A, t_end_A = build_scheduled_vehicle(
    "scheduled_a", "Peccioli → Pontedera", peak_am, scheduled_coords, reverse=False
)
vehicles_weekday.append(veh_sched_A)

# Scheduled B: pontedera_1 → peccioli_1 alle 8:00 (parte in direzione opposta)
veh_sched_B_am, _ = build_scheduled_vehicle(
    "scheduled_b", "Pontedera → Peccioli (mattina)", peak_am, scheduled_coords, reverse=True
)
vehicles_weekday.append(veh_sched_B_am)

# Scheduled A ritorno: parte da pontedera a peak_pm
veh_sched_A_pm, _ = build_scheduled_vehicle(
    "scheduled_a", "Pontedera → Peccioli (sera)", peak_pm, scheduled_coords, reverse=True
)
# Merge waypoints nella stessa entry scheduled_a
vehicles_weekday[0]["waypoints"] += veh_sched_A_pm["waypoints"]

# Scheduled B ritorno a peak_pm
veh_sched_B_pm, _ = build_scheduled_vehicle(
    "scheduled_b", "Peccioli → Pontedera (sera)", peak_pm, scheduled_coords, reverse=False
)
vehicles_weekday[1]["waypoints"] += veh_sched_B_pm["waypoints"]

# Bacini on-demand
for i, bacino in enumerate(ON_DEMAND_BACINI):
    if bacino in bacini:
        v = build_ondemand_vehicle(bacino, peak_am, peak_pm, seed_offset=i)
        vehicles_weekday.append(v)
        print(f"  ✓ {bacino}: {len(v['waypoints'])} waypoints")

weekday_scenario = {"scenario": "weekday", "vehicles": vehicles_weekday}
print(f"\nWeekday: {len(vehicles_weekday)} veicoli")

  ✓ peccioli: 1598 waypoints
  ✓ forcoli: 1287 waypoints
  ✓ fabbrica: 741 waypoints
  ✓ belvedere: 764 waypoints
  ✓ rosa: 905 waypoints
  ✓ ghizzano: 587 waypoints

Weekday: 8 veicoli


In [22]:
# ── SCENARIO WEEKEND ─────────────────────────────────────────────────────────

peak_am = WEEKEND_PEAK_AM
peak_pm = WEEKEND_PEAK_PM

vehicles_weekend = []

# Scheduled A e B
veh_sched_A, _ = build_scheduled_vehicle(
    "scheduled_a", "Peccioli → Pontedera", peak_am, scheduled_coords, reverse=False
)
veh_sched_B_am, _ = build_scheduled_vehicle(
    "scheduled_b", "Pontedera → Peccioli (mattina)", peak_am, scheduled_coords, reverse=True
)
veh_sched_A_pm, _ = build_scheduled_vehicle(
    "scheduled_a", "Pontedera → Peccioli (sera)", peak_pm, scheduled_coords, reverse=True
)
veh_sched_B_pm, _ = build_scheduled_vehicle(
    "scheduled_b", "Peccioli → Pontedera (sera)", peak_pm, scheduled_coords, reverse=False
)
veh_sched_A["waypoints"] += veh_sched_A_pm["waypoints"]
veh_sched_B_am["waypoints"] += veh_sched_B_pm["waypoints"]
vehicles_weekend.extend([veh_sched_A, veh_sched_B_am])

# Bacini on-demand
for i, bacino in enumerate(ON_DEMAND_BACINI):
    if bacino in bacini:
        v = build_ondemand_vehicle(bacino, peak_am, peak_pm, seed_offset=i+10)
        vehicles_weekend.append(v)
        print(f"  ✓ {bacino}: {len(v['waypoints'])} waypoints")

weekend_scenario = {"scenario": "weekend", "vehicles": vehicles_weekend}
print(f"\nWeekend: {len(vehicles_weekend)} veicoli")

  ✓ peccioli: 1312 waypoints
  ✓ forcoli: 1213 waypoints
  ✓ fabbrica: 679 waypoints
  ✓ belvedere: 729 waypoints
  ✓ rosa: 1011 waypoints
  ✓ ghizzano: 707 waypoints

Weekend: 8 veicoli


In [23]:
# ── SCENARIO EVENTDAY ────────────────────────────────────────────────────────
# Scheduled: orari weekday (8:00 e 18:00)
# Bacini: aggiungono viaggio peccioli_1 → belvedere_5 alle 12:00 e ritorno alle 17:00
# poi continuano on-demand normale

peak_am = WEEKDAY_PEAK_AM
peak_pm = WEEKDAY_PEAK_PM

vehicles_event = []

# Scheduled identico al weekday
veh_sched_A, _ = build_scheduled_vehicle(
    "scheduled_a", "Peccioli → Pontedera", peak_am, scheduled_coords, reverse=False
)
veh_sched_B_am, _ = build_scheduled_vehicle(
    "scheduled_b", "Pontedera → Peccioli (mattina)", peak_am, scheduled_coords, reverse=True
)
veh_sched_A_pm, _ = build_scheduled_vehicle(
    "scheduled_a", "Pontedera → Peccioli (sera)", peak_pm, scheduled_coords, reverse=True
)
veh_sched_B_pm, _ = build_scheduled_vehicle(
    "scheduled_b", "Peccioli → Pontedera (sera)", peak_pm, scheduled_coords, reverse=False
)
veh_sched_A["waypoints"] += veh_sched_A_pm["waypoints"]
veh_sched_B_am["waypoints"] += veh_sched_B_pm["waypoints"]
vehicles_event.extend([veh_sched_A, veh_sched_B_am])

# Bacini: costruiamo on-demand normale poi inseriamo evento
for i, bacino in enumerate(ON_DEMAND_BACINI):
    if bacino not in bacini:
        continue

    # Prima costruiamo la giornata normale
    v = build_ondemand_vehicle(bacino, peak_am, peak_pm, seed_offset=i+20)

    # ── Trip evento: peccioli_1 → belvedere_5 alle 12:00 ──────────────────
    wps_to_bel, t_at_bel = build_timed_waypoints(
        G, node_latlon, [HUB, BEL5], EVENT_TRIP1, SPEED_MS
    )

    # ── Ritorno belvedere_5 → peccioli_1 poi on-demand pomeridiano ────────
    wps_from_bel, t_back_hub = build_timed_waypoints(
        G, node_latlon, [BEL5, HUB], EVENT_TRIP2, SPEED_MS
    )

    # Inseriamo l'evento nella timeline: troviamo i waypoint tra 11:30 e 17:30
    # e li sostituiamo con l'evento (i veicoli si fermano a 11:55, poi evento)
    T_START_EVENT = EVENT_TRIP1 - 300  # 5 min prima si fermano
    T_END_EVENT   = t_back_hub

    # Waypoints prima dell'evento
    wps_before = [w for w in v["waypoints"] if w["t"] < T_START_EVENT]
    # Aggiungi punto di attesa all'hub
    if wps_before:
        last = wps_before[-1]
        wps_before.append({"t": EVENT_TRIP1 - 60, "lat": HUB[0], "lng": HUB[1]})

    # Waypoints dopo l'evento
    wps_after = [w for w in v["waypoints"] if w["t"] > T_END_EVENT]

    # Assembla
    v["waypoints"] = wps_before + wps_to_bel + wps_from_bel + wps_after
    vehicles_event.append(v)
    print(f"  ✓ {bacino}: {len(v['waypoints'])} waypoints (con evento)")

eventday_scenario = {"scenario": "eventday", "vehicles": vehicles_event}
print(f"\nEventday: {len(vehicles_event)} veicoli")

  ✓ peccioli: 823 waypoints (con evento)
  ✓ forcoli: 804 waypoints (con evento)
  ✓ fabbrica: 547 waypoints (con evento)
  ✓ belvedere: 620 waypoints (con evento)
  ✓ rosa: 719 waypoints (con evento)
  ✓ ghizzano: 609 waypoints (con evento)

Eventday: 8 veicoli


In [24]:
# ── ESPORTA JSON ─────────────────────────────────────────────────────────────

import os
os.makedirs(OUT_DIR, exist_ok=True)

for scenario, fname in [
    (weekday_scenario,  "weekday.json"),
    (weekend_scenario,  "weekend.json"),
    (eventday_scenario, "eventday.json"),
]:
    path = os.path.join(OUT_DIR, fname)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(scenario, f, ensure_ascii=False, separators=(",", ":"))
    size_kb = os.path.getsize(path) / 1024
    print(f"  ✓ {fname} — {size_kb:.1f} KB")

print("\n✅ Tutti gli scenari esportati. Copia la cartella data/scenarios/ nel repo GitHub.")

  ✓ weekday.json — 353.3 KB
  ✓ weekend.json — 342.1 KB
  ✓ eventday.json — 264.6 KB

✅ Tutti gli scenari esportati. Copia la cartella data/scenarios/ nel repo GitHub.


In [25]:
# ── DEBUG: visualizza waypoints di un veicolo ────────────────────────────────
# Cambia il nome del veicolo per ispezionare

VEH_TO_INSPECT = "rosa"  # id del veicolo
SCENARIO = weekday_scenario

veh = next((v for v in SCENARIO["vehicles"] if v["id"] == VEH_TO_INSPECT), None)
if veh:
    print(f"Veicolo: {veh['id']} | {len(veh['waypoints'])} waypoints")
    print(f"Inizio: {veh['waypoints'][0]}")
    print(f"Fine:   {veh['waypoints'][-1]}")
    for wp in veh['waypoints'][:5]:
        t = wp['t']
        print(f"  t={t//3600:02d}:{(t%3600)//60:02d}:{t%60:02d}  lat={wp['lat']:.5f}  lng={wp['lng']:.5f}")
    print("  ...")
else:
    print(f"Veicolo '{VEH_TO_INSPECT}' non trovato")

Veicolo: rosa | 905 waypoints
Inizio: {'t': 27962, 'lat': np.float64(43.5466074), 'lng': np.float64(10.714509)}
Fine:   {'t': 67947, 'lat': 43.5292237, 'lng': 10.6694718}
  t=07:46:02  lat=43.54661  lng=10.71451
  t=07:46:16  lat=43.54487  lng=10.71494
  t=07:46:30  lat=43.54378  lng=10.71483
  t=07:46:44  lat=43.54337  lng=10.71226
  t=07:46:59  lat=43.54314  lng=10.71056
  ...
